# PropIQ - Week 10
## Controlled Streaming Simulation: SQL-First Auto Loader Notebook

**ZENAIZ x BVRIT Hyderabad Data Engineering Internship | P15 PropIQ: Real Estate Market Analytics**  
**Notebook path:** `notebooks/07_streaming_simulation.ipynb`  
**Technology:** Databricks Free Edition (serverless) | Spark SQL first | light PySpark | Auto Loader | Delta

> **Week 10 job:** process all six listing-status drops incrementally with an explicit schema, a checkpoint, a two-day watermark, `listing_event_id` de-duplication, sequence validation, quarantine and schema-drift rescue, and prove every control with SQL.

### Coding principle (same as PageLoop Weeks 5, 6 and 10)

Use **SQL wherever SQL is the clearest choice**: listing files, inspecting JSON, creating tables, routing rules, reconciliation, Delta history and checkpoint state.

Use **light PySpark only for the streaming plumbing**: the Auto Loader source and the `foreachBatch` write that runs the SQL rules.

## 1. Outcome first - what will PropIQ produce?

Three streaming Delta tables, exactly as named in the playbook table contract:

| Layer | Table | Responsibility |
|---|---|---|
| Streaming Bronze | `workspace.default.bronze_listing_status_stream` | every physical record that arrived, raw-preserving, with lineage |
| Streaming Trusted Silver | `workspace.default.silver_listing_status_stream_trusted` | accepted events only |
| Streaming Quarantine | `workspace.default.quarantine_listing_status_stream` | every failed physical record with rule ID, reasons, severity and lineage |

```text
6 JSON Lines drops  (source/)
        |  copied ONE AT A TIME
        v
landing/  --> Auto Loader (explicit schema, rescued data, 2-day watermark declared)
        |             \
        |              checkpoint:  checkpoints/listing_status_event
        v
foreachBatch  -->  bronze_listing_status_stream            (raw, nothing dropped)
        |
        +--> SQL rules --> silver_listing_status_stream_trusted
                       \-> quarantine_listing_status_stream
```

### What you must prove

- Each drop is processed once, in order, without reprocessing old files.
- A repeated `listing_event_id` is not double-counted.
- A late event (older than the watermark) is **retained**, not silently lost.
- Malformed, orphan, invalid and future-dated events are retained with a reason.
- An unknown field is rescued and flagged, not promoted automatically.
- A run with no new file changes nothing.
- `Bronze physical rows = Trusted + Quarantine` for every file.

> No mentor-only expected counts are copied into this notebook. Every number you report must come from **your own Databricks run**.

## 2. Design decisions you must be able to defend

| Decision | Why |
|---|---|
| **Bronze keeps everything, including bad lines** | Bronze is evidence. Nothing is filtered or de-duplicated there. |
| **The watermark is enforced in SQL, not with `dropDuplicates` state** | Spark's stateful watermark *silently discards* late rows. The playbook requires late events to be **retained** in quarantine, so the rule is written explicitly. |
| **Only accepted events move the watermark** | A single future-dated event (for example year 2027) must not drag the watermark forward and make good events look late. |
| **Rules collect all applicable reasons** | The playbook forbids first-failure-only routing. A record can be both late and a duplicate. |
| **`foreachBatch` with idempotent Delta writes** | Each batch writes with `txnAppId` / `txnVersion`, so a retried batch is not appended twice. |
| **Quarantine is written before Silver** | The rules treat Silver as "already accepted". Writing Silver first would make a batch see its own rows as duplicates. |
| **Parents come from Trusted Silver** | Orphan checks use `silver_listings_trusted`, `silver_localities_trusted` and `silver_brokers_trusted`, never raw Bronze. |

## 3. The six drops - what each one is meant to test

| Drop | File | Scenario | What the stream should do |
|---|---|---|---|
| 01 | `listing_status_event_drop_01_normal.json` | normal baseline | accept events; establish event shape and sequence |
| 02 | `listing_status_event_drop_02_duplicate.json` | repeated event ID | keep one copy; quarantine the repeat |
| 03 | `listing_status_event_drop_03_late_out_of_order.json` | late and out-of-order | quarantine the late event and the sequence violation |
| 04 | `listing_status_event_drop_04_malformed_reference.json` | malformed JSON and orphan reference | retain the raw bad line; quarantine the orphan |
| 05 | `listing_status_event_drop_05_invalid_payload_future.json` | invalid price/area or future time | quarantine; watermark must not move |
| 06 | `listing_status_event_drop_06_schema_drift.json` | extra field | rescue the unknown field; accept with a drift flag; do not promote |

The "should do" column comes from the sample rows printed in the PropIQ Student Playbook. The real files may contain more rows, so always read your own query output instead of assuming.

## 4. Locked streaming controls and where each one lives

| Control | Locked behaviour (playbook) | Implemented by |
|---|---|---|
| Input | six incremental JSON Lines drops | copy `source/` to `landing/` one file at a time |
| Checkpoint | `.../checkpoints/listing_status_event` | `checkpointLocation` of the stream |
| Watermark | 2 days on `event_timestamp` | `.withWatermark("event_timestamp", "2 days")` plus the SQL rule `event_timestamp < MAX(accepted event_timestamp) - 2 days` |
| Deduplication | `listing_event_id` | reason `duplicate_listing_event_id` |
| Ordering | `event_sequence` increases within `listing_id` | reason `sequence_violation` |
| Malformed | `_rescued_data` or `badRecordsPath`, then quarantine | `_corrupt_record` and `_rescued_data` kept in Bronze, routed to quarantine |
| Late | `late_event_quarantine` | reason code `late_event_quarantine` |
| Schema drift | rescue unknown fields; promote only after review | `_rescued_data` kept, `schema_drift_flag` set, no automatic promotion |

## 5. Fixed sample paths and objects

Your workspace volume is `workspace.default.propqi_streaming`.

| Purpose | Path |
|---|---|
| six source drops | `/Volumes/workspace/default/propqi_streaming/source/` |
| landing folder (starts empty) | `/Volumes/workspace/default/propqi_streaming/landing/` |
| checkpoint | `/Volumes/workspace/default/propqi_streaming/checkpoints/listing_status_event` |
| Streaming Bronze | `workspace.default.bronze_listing_status_stream` |
| Streaming Trusted Silver | `workspace.default.silver_listing_status_stream_trusted` |
| Streaming Quarantine | `workspace.default.quarantine_listing_status_stream` |

> The playbook writes the checkpoint as `/Volumes/<catalog>/<schema>/propiq/checkpoints/listing_status_event`. Here the volume segment is your `propqi_streaming` volume. If your volume is actually spelled `propiq_streaming`, find-and-replace `propqi_streaming` once across this notebook before running anything.

## 6. Step 1 - confirm the working location and the previous-week dependency

Week 10 builds on accepted earlier work. The orphan rules need the Trusted Silver masters from the batch weeks. If any is missing, **stop and record a blocker in `weekly_logs/week10_log.md`**. Do not continue downstream.

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA default;

SELECT
  current_catalog() AS active_catalog,
  current_schema()  AS active_schema;

active_catalog,active_schema
workspace,default


**Expected result:** `workspace` and `default`.

### 6.1 Confirm the three parent masters exist and have rows

In [0]:
%sql
SELECT 'silver_listings_trusted'   AS parent_table, COUNT(*) AS rows FROM workspace.default.silver_listings_trusted
UNION ALL
SELECT 'silver_localities_trusted' AS parent_table, COUNT(*) AS rows FROM workspace.default.silver_localities_trusted
UNION ALL
SELECT 'silver_brokers_trusted'    AS parent_table, COUNT(*) AS rows FROM workspace.default.silver_brokers_trusted;

parent_table,rows
silver_listings_trusted,49000
silver_localities_trusted,80
silver_brokers_trusted,320


**Expected result:** three rows, each with a non-zero count.

If a table is not found, the batch pipeline is not in `workspace.default`, or an earlier week is still open. Fix that first.

## 7. Step 2 - create the volume folders and place the six drops in `source/`

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.default.propqi_streaming;

In [0]:
%fs mkdirs /Volumes/workspace/default/propqi_streaming/source/

True

In [0]:
%fs mkdirs /Volumes/workspace/default/propqi_streaming/landing/

True

In [0]:
%fs mkdirs /Volumes/workspace/default/propqi_streaming/checkpoints/

True

Upload the six files from the repository folder `data_sample/streaming/` into:

`/Volumes/workspace/default/propqi_streaming/source/`

Do **not** put them in `landing/`. They are copied there one at a time later.

In [0]:
%sql
LIST '/Volumes/workspace/default/propqi_streaming/source/';

path,name,size,modification_time
/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_01_normal.json,listing_status_event_drop_01_normal.json,1725,1791054438000
/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_02_duplicate.json,listing_status_event_drop_02_duplicate.json,1290,1791054438000
/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_03_late_out_of_order.json,listing_status_event_drop_03_late_out_of_order.json,1285,1791054438000
/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_04_malformed_reference.json,listing_status_event_drop_04_malformed_reference.json,898,1791054439000
/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_05_invalid_payload_future.json,listing_status_event_drop_05_invalid_payload_future.json,1291,1791054439000
/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_06_schema_drift.json,listing_status_event_drop_06_schema_drift.json,1322,1791054438000


**Expected result:** all six drop files are listed.

In [0]:
%sql
LIST '/Volumes/workspace/default/propqi_streaming/landing/';

path,name,size,modification_time


**Expected result:** no files. The landing folder must be empty before Drop 01 arrives.

## 8. Step 3 - inspect the raw source with SQL before streaming

`read_files` lets us look at each JSON Lines file directly in SQL.

### 8.1 Physical line count per source file (control total)

Reading as plain text counts physical lines, so this does not depend on JSON parsing. Blank lines are ignored.

In [0]:
%sql
SELECT 'listing_status_event_drop_01_normal.json' AS source_file_name, COUNT(*) AS source_lines
FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_01_normal.json', format => 'text') WHERE trim(value) <> ''
UNION ALL
SELECT 'listing_status_event_drop_02_duplicate.json', COUNT(*)
FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_02_duplicate.json', format => 'text') WHERE trim(value) <> ''
UNION ALL
SELECT 'listing_status_event_drop_03_late_out_of_order.json', COUNT(*)
FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_03_late_out_of_order.json', format => 'text') WHERE trim(value) <> ''
UNION ALL
SELECT 'listing_status_event_drop_04_malformed_reference.json', COUNT(*)
FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_04_malformed_reference.json', format => 'text') WHERE trim(value) <> ''
UNION ALL
SELECT 'listing_status_event_drop_05_invalid_payload_future.json', COUNT(*)
FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_05_invalid_payload_future.json', format => 'text') WHERE trim(value) <> ''
UNION ALL
SELECT 'listing_status_event_drop_06_schema_drift.json', COUNT(*)
FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_06_schema_drift.json', format => 'text') WHERE trim(value) <> ''
ORDER BY source_file_name;

source_file_name,source_lines
listing_status_event_drop_01_normal.json,4
listing_status_event_drop_02_duplicate.json,3
listing_status_event_drop_03_late_out_of_order.json,3
listing_status_event_drop_04_malformed_reference.json,3
listing_status_event_drop_05_invalid_payload_future.json,3
listing_status_event_drop_06_schema_drift.json,3


**Record these six numbers.** They are your source control totals and are compared with Bronze at the end.

### 8.2 Look at one normal drop

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_01_normal.json',
  format => 'json'
)
LIMIT 10;

area_sqft,asking_price,bedrooms,broker_id,event_sequence,event_timestamp,event_type,ingestion_timestamp,listing_event_id,listing_id,listing_status,locality_id,property_type,schema_version,source_system,_rescued_data
1631,11005988,4,BRK-0191,1,2026-06-25T09:00:00,created,2026-06-25T09:02:00,EVT-000001,LST-0000001,active,LOC-062,Apartment,1.0,PROPIQ_EVENT_SIM,null
1157,21357386,2,BRK-0130,2,2026-06-25T09:10:00,price_changed,2026-06-25T09:12:00,EVT-000002,LST-0000002,active,LOC-056,Apartment,1.0,PROPIQ_EVENT_SIM,null
1973,16873096,1,BRK-0215,2,2026-06-25T09:20:00,paused,2026-06-25T09:22:00,EVT-000003,LST-0000003,paused,LOC-045,Apartment,1.0,PROPIQ_EVENT_SIM,null
1164,21144060,4,BRK-0177,3,2026-06-25T09:30:00,reactivated,2026-06-25T09:32:00,EVT-000004,LST-0000004,active,LOC-036,Apartment,1.0,PROPIQ_EVENT_SIM,null


One row means: **one status event for one listing at one event time**. `listing_event_id` identifies the event; `listing_id` identifies the listing it belongs to.

### 8.3 Dry-run reference check on Drop 01

Before any stream runs, confirm that Drop 01 references resolve to the Trusted Silver masters. If almost everything is unresolved, the masters and the events do not belong together. That is a blocker, not something to hide by loosening a rule.

In [0]:
%sql
SELECT
  COUNT(*)                                                                    AS events_in_drop_01,
  SUM(CASE WHEN l.k IS NULL THEN 1 ELSE 0 END)                                AS listing_not_in_trusted_master,
  SUM(CASE WHEN c.k IS NULL THEN 1 ELSE 0 END)                                AS locality_not_in_trusted_master,
  SUM(CASE WHEN b.k IS NULL THEN 1 ELSE 0 END)                                AS broker_not_in_trusted_master
FROM read_files(
       '/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_01_normal.json',
       format => 'json',
       schema => 'listing_event_id STRING, listing_id STRING, locality_id STRING, broker_id STRING'
     ) e
LEFT JOIN (SELECT DISTINCT listing_id  AS k FROM workspace.default.silver_listings_trusted)   l ON e.listing_id  = l.k
LEFT JOIN (SELECT DISTINCT locality_id AS k FROM workspace.default.silver_localities_trusted) c ON e.locality_id = c.k
LEFT JOIN (SELECT DISTINCT broker_id   AS k FROM workspace.default.silver_brokers_trusted)    b ON e.broker_id   = b.k;

events_in_drop_01,listing_not_in_trusted_master,locality_not_in_trusted_master,broker_not_in_trusted_master
4,0,0,0


**Expected behaviour:** a normal baseline drop should resolve against the masters. Large unresolved counts must be explained in the Week Log before you continue.

## 9. Step 4 - create the three target tables with an explicit schema

Creating the tables first (empty) has two benefits: the schema is explicit, and the rules can safely read "what has already been accepted" even on the very first batch.

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.default.bronze_listing_status_stream (
  listing_event_id      STRING,
  listing_id            STRING,
  event_type            STRING,
  event_timestamp       TIMESTAMP,
  event_sequence        INT,
  listing_status        STRING,
  asking_price          BIGINT,
  locality_id           STRING,
  broker_id             STRING,
  property_type         STRING,
  bedrooms              INT,
  area_sqft             INT,
  source_system         STRING,
  schema_version        STRING,
  ingestion_timestamp   TIMESTAMP,
  _corrupt_record       STRING,
  _rescued_data         STRING,
  source_file_name      STRING,
  source_file_path      STRING,
  record_uid            STRING,
  batch_id              BIGINT,
  bronze_ingested_at    TIMESTAMP
)
COMMENT 'PropIQ Week 10 streaming Bronze: every physical listing-status record, raw-preserving';

CREATE TABLE IF NOT EXISTS workspace.default.silver_listing_status_stream_trusted (
  record_uid            STRING,
  listing_event_id      STRING,
  listing_id            STRING,
  event_type            STRING,
  event_timestamp       TIMESTAMP,
  event_sequence        INT,
  listing_status        STRING,
  asking_price          BIGINT,
  locality_id           STRING,
  broker_id             STRING,
  property_type         STRING,
  bedrooms              INT,
  area_sqft             INT,
  source_system         STRING,
  schema_version        STRING,
  ingestion_timestamp   TIMESTAMP,
  schema_drift_flag     BOOLEAN,
  rescued_data          STRING,
  watermark_ts_applied  TIMESTAMP,
  source_file_name      STRING,
  batch_id              BIGINT,
  processed_at          TIMESTAMP
)
COMMENT 'PropIQ Week 10 streaming Trusted Silver: accepted listing-status events only';

CREATE TABLE IF NOT EXISTS workspace.default.quarantine_listing_status_stream (
  record_uid            STRING,
  listing_event_id      STRING,
  listing_id            STRING,
  event_type            STRING,
  event_timestamp       TIMESTAMP,
  event_sequence        INT,
  listing_status        STRING,
  asking_price          BIGINT,
  locality_id           STRING,
  broker_id             STRING,
  property_type         STRING,
  bedrooms              INT,
  area_sqft             INT,
  source_system         STRING,
  schema_version        STRING,
  ingestion_timestamp   TIMESTAMP,
  raw_record            STRING,
  rescued_data          STRING,
  failure_reasons       ARRAY<STRING>,
  primary_reason        STRING,
  rule_id               STRING,
  severity              STRING,
  watermark_ts_applied  TIMESTAMP,
  source_file_name      STRING,
  batch_id              BIGINT,
  quarantined_at        TIMESTAMP
)
COMMENT 'PropIQ Week 10 streaming quarantine: every failed physical record with rule evidence';

In [0]:
%sql
SELECT 'bronze_listing_status_stream'            AS table_name, COUNT(*) AS rows FROM workspace.default.bronze_listing_status_stream
UNION ALL
SELECT 'silver_listing_status_stream_trusted'    AS table_name, COUNT(*) AS rows FROM workspace.default.silver_listing_status_stream_trusted
UNION ALL
SELECT 'quarantine_listing_status_stream'        AS table_name, COUNT(*) AS rows FROM workspace.default.quarantine_listing_status_stream;

table_name,rows
bronze_listing_status_stream,0
silver_listing_status_stream_trusted,0
quarantine_listing_status_stream,0


**Expected result:** three tables, all with `0` rows before Drop 01.

## 10. Step 5 - define the Auto Loader source

This is one of the two places where light PySpark is clearer. Read it in plain English:

> Watch `landing/`. Parse each file with an **explicit schema**. Keep anything that does not fit in `_rescued_data`. Keep an unparseable line in `_corrupt_record`. Declare a 2-day watermark on `event_timestamp`.

Why the explicit schema matters: Auto Loader never guesses types. A field that does not match the contract is rescued, not silently re-typed.

In [0]:
%python
from pyspark.sql import functions as F

# ---- fixed PropIQ Week 10 locations --------------------------------------------------
LANDING    = "/Volumes/workspace/default/propqi_streaming/landing/"
CHECKPOINT = "/Volumes/workspace/default/propqi_streaming/checkpoints/listing_status_event"
BRONZE     = "workspace.default.bronze_listing_status_stream"
SILVER     = "workspace.default.silver_listing_status_stream_trusted"
QUARANTINE = "workspace.default.quarantine_listing_status_stream"

WATERMARK_DELAY = "2 days"          # locked control: 2 days on event_timestamp

# ---- explicit schema (the streaming event contract) ----------------------------------
EVENT_SCHEMA = (
    "listing_event_id STRING, "
    "listing_id STRING, "
    "event_type STRING, "
    "event_timestamp TIMESTAMP, "
    "event_sequence INT, "
    "listing_status STRING, "
    "asking_price BIGINT, "
    "locality_id STRING, "
    "broker_id STRING, "
    "property_type STRING, "
    "bedrooms INT, "
    "area_sqft INT, "
    "source_system STRING, "
    "schema_version STRING, "
    "ingestion_timestamp TIMESTAMP, "
    "_corrupt_record STRING"
)

events_stream = (
    spark.readStream
         .format("cloudFiles")
         .option("cloudFiles.format", "json")
         .option("rescuedDataColumn", "_rescued_data")           # unknown fields and type mismatches
         .option("columnNameOfCorruptRecord", "_corrupt_record")  # unparseable lines
         .schema(EVENT_SCHEMA)
         .load(LANDING)
         .select(
             "*",
             F.col("_metadata.file_name").alias("source_file_name"),
             F.col("_metadata.file_path").alias("source_file_path"),
         )
         .withWatermark("event_timestamp", WATERMARK_DELAY)
)

print("Auto Loader source is defined. Nothing has been read or written yet.")

Auto Loader source is defined. Nothing has been read or written yet.


**Checkpoint:** Auto Loader watches `landing/`, not `source/`.

> `withWatermark` declares the tolerance in the stream definition. A stateless `foreachBatch` does not drop rows by itself, which is exactly why late rows can be retained. The rule that **enforces** the two-day window is the SQL in the next step, using the same constant.

## 11. Step 6 - the routing rules, in SQL

Every physical Bronze row of a batch is evaluated by the SQL below and receives a `route` of `trusted` or `quarantine`.

| Order | Reason code | Rule ID | Severity | Meaning |
|---:|---|---|---|---|
| 1 | `malformed_json` | P15-STR-01 | Critical | line could not be parsed |
| 2 | `missing_event_key` | P15-DQ-01 | Critical | key field is null |
| 3 | `orphan_listing_reference` | P15-DQ-02 | Critical | `listing_id` not in `silver_listings_trusted` |
| 4 | `orphan_locality_reference` | P15-DQ-02 | Critical | `locality_id` not in `silver_localities_trusted` |
| 5 | `orphan_broker_reference` | P15-DQ-02 | Critical | `broker_id` not in `silver_brokers_trusted` |
| 6 | `invalid_price` | P15-DQ-03 | Major | price outside 2M to 150M INR |
| 7 | `invalid_area` | P15-DQ-03 | Major | area outside 300 to 8,000 sq ft |
| 8 | `future_event_time` | P15-STR-02 | Major | event time after now, or after the simulation horizon `2026-12-31` |
| 9 | `duplicate_listing_event_id` | P15-STR-03 | Major | same `listing_event_id` already accepted or repeated in the batch |
| 10 | `late_event_quarantine` | P15-STR-04 | Major | event time older than `MAX(accepted event_timestamp) - 2 days` |
| 11 | `sequence_violation` | P15-STR-05 | Major | `event_sequence` does not increase within `listing_id` |

Notes you should be able to explain:

- `P15-DQ-*` IDs come from the playbook Data Quality Catalogue. The `P15-STR-*` IDs are **notebook-local** codes for stream-only controls; list them in `streaming/structured_streaming_design.md`.
- Rules 9 to 11 are stateful, so only rows that passed rules 1 to 8 compete for them. A bad row must never become the "winner" of a duplicate or set the watermark.
- The first batch has no accepted history, so no watermark exists and nothing can be late.
- Schema drift is **not** a failure. A row with a rescued field (or `schema_version` other than `1.0`) is accepted with `schema_drift_flag = true`.

In [0]:
%python
CLASSIFY_SQL = """
CREATE OR REPLACE TEMP VIEW stream_batch_classified AS
WITH
-- 1. Watermark = latest ACCEPTED event time from earlier batches - 2 days.
--    Only accepted events move it, so one bad future timestamp cannot drag it forward.
wm AS (
  SELECT MAX(event_timestamp) - INTERVAL 2 DAYS AS watermark_ts
  FROM workspace.default.silver_listing_status_stream_trusted
),

-- 2. Attach watermark and parent-key lookups to every physical Bronze row of this batch.
joined AS (
  SELECT
    b.*,
    wm.watermark_ts,
    (b._corrupt_record IS NOT NULL
      OR (b.listing_event_id IS NULL AND b.listing_id IS NULL AND b.event_type IS NULL)) AS f_malformed,
    (pl.k IS NULL) AS no_listing,
    (pc.k IS NULL) AS no_locality,
    (pb.k IS NULL) AS no_broker
  FROM stg_batch b
  CROSS JOIN wm
  LEFT JOIN (SELECT DISTINCT listing_id  AS k FROM workspace.default.silver_listings_trusted)   pl ON b.listing_id  = pl.k
  LEFT JOIN (SELECT DISTINCT locality_id AS k FROM workspace.default.silver_localities_trusted) pc ON b.locality_id = pc.k
  LEFT JOIN (SELECT DISTINCT broker_id   AS k FROM workspace.default.silver_brokers_trusted)    pb ON b.broker_id   = pb.k
),

-- 3. Row-level rules that need no state.
flagged AS (
  SELECT
    j.*,
    (NOT j.f_malformed AND (j.listing_event_id IS NULL OR j.listing_id IS NULL
                            OR j.event_timestamp IS NULL OR j.event_sequence IS NULL))        AS f_missing_key,
    (NOT j.f_malformed AND j.listing_id  IS NOT NULL AND j.no_listing)                         AS f_orphan_listing,
    (NOT j.f_malformed AND j.locality_id IS NOT NULL AND j.no_locality)                        AS f_orphan_locality,
    (NOT j.f_malformed AND j.broker_id   IS NOT NULL AND j.no_broker)                          AS f_orphan_broker,
    (NOT j.f_malformed AND NOT COALESCE(j.asking_price BETWEEN 2000000 AND 150000000, FALSE))  AS f_invalid_price,
    (NOT j.f_malformed AND NOT COALESCE(j.area_sqft BETWEEN 300 AND 8000, FALSE))              AS f_invalid_area,
    COALESCE(NOT j.f_malformed
             AND (j.event_timestamp > current_timestamp()
                  OR j.event_timestamp > TIMESTAMP '2026-12-31 23:59:59'), FALSE)              AS f_future
  FROM joined j
),

-- 4. Only rows that passed every row-level rule are allowed to compete for dedup / late / sequence.
eligible AS (
  SELECT * FROM flagged
  WHERE NOT (f_malformed OR f_missing_key OR f_orphan_listing OR f_orphan_locality
             OR f_orphan_broker OR f_invalid_price OR f_invalid_area OR f_future)
),

-- 5. Duplicate (same listing_event_id) and late (older than the watermark).
dedup AS (
  SELECT
    e.record_uid, e.listing_id, e.listing_event_id, e.event_timestamp, e.event_sequence,
    (ROW_NUMBER() OVER (PARTITION BY e.listing_event_id
                        ORDER BY e.event_timestamp, e.ingestion_timestamp, e.record_uid) > 1
     OR s.k IS NOT NULL)                                          AS f_duplicate,
    COALESCE(e.event_timestamp < e.watermark_ts, FALSE)           AS f_late
  FROM eligible e
  LEFT JOIN (SELECT DISTINCT listing_event_id AS k
             FROM workspace.default.silver_listing_status_stream_trusted) s
         ON e.listing_event_id = s.k
),

-- 6. Sequence control: event_sequence must increase within listing_id, in event-time order.
ordered AS (
  SELECT
    d.*,
    MAX(d.event_sequence) OVER (PARTITION BY d.listing_id
                                ORDER BY d.event_timestamp, d.listing_event_id
                                ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS prev_seq_in_batch
  FROM dedup d
  WHERE NOT d.f_duplicate AND NOT d.f_late
),
sequenced AS (
  SELECT
    o.record_uid,
    (o.event_sequence <= GREATEST(COALESCE(o.prev_seq_in_batch, -1), COALESCE(h.max_seq, -1))) AS f_sequence
  FROM ordered o
  LEFT JOIN (SELECT listing_id, MAX(event_sequence) AS max_seq
             FROM workspace.default.silver_listing_status_stream_trusted
             GROUP BY listing_id) h
         ON o.listing_id = h.listing_id
),

-- 7. Put every flag back on the physical row, then collect ALL applicable reasons (not first-failure-only).
final AS (
  SELECT
    f.*,
    COALESCE(d.f_duplicate, FALSE) AS f_duplicate,
    COALESCE(d.f_late, FALSE)      AS f_late,
    COALESCE(q.f_sequence, FALSE)  AS f_sequence
  FROM flagged f
  LEFT JOIN dedup     d ON f.record_uid = d.record_uid
  LEFT JOIN sequenced q ON f.record_uid = q.record_uid
),
reasons AS (
  SELECT
    x.*,
    FILTER(ARRAY(
      IF(x.f_malformed,        'malformed_json',             NULL),
      IF(x.f_missing_key,      'missing_event_key',          NULL),
      IF(x.f_orphan_listing,   'orphan_listing_reference',   NULL),
      IF(x.f_orphan_locality,  'orphan_locality_reference',  NULL),
      IF(x.f_orphan_broker,    'orphan_broker_reference',    NULL),
      IF(x.f_invalid_price,    'invalid_price',              NULL),
      IF(x.f_invalid_area,     'invalid_area',               NULL),
      IF(x.f_future,           'future_event_time',          NULL),
      IF(x.f_duplicate,        'duplicate_listing_event_id', NULL),
      IF(x.f_late,             'late_event_quarantine',      NULL),
      IF(x.f_sequence,         'sequence_violation',         NULL)
    ), r -> r IS NOT NULL) AS failure_reasons
  FROM final x
)
SELECT
  r.*,
  IF(SIZE(r.failure_reasons) = 0, 'trusted', 'quarantine') AS route,
  r.failure_reasons[0]                                      AS primary_reason,
  CASE r.failure_reasons[0]
    WHEN 'malformed_json'             THEN 'P15-STR-01'
    WHEN 'missing_event_key'          THEN 'P15-DQ-01'
    WHEN 'orphan_listing_reference'   THEN 'P15-DQ-02'
    WHEN 'orphan_locality_reference'  THEN 'P15-DQ-02'
    WHEN 'orphan_broker_reference'    THEN 'P15-DQ-02'
    WHEN 'invalid_price'              THEN 'P15-DQ-03'
    WHEN 'invalid_area'               THEN 'P15-DQ-03'
    WHEN 'future_event_time'          THEN 'P15-STR-02'
    WHEN 'duplicate_listing_event_id' THEN 'P15-STR-03'
    WHEN 'late_event_quarantine'      THEN 'P15-STR-04'
    WHEN 'sequence_violation'         THEN 'P15-STR-05'
  END AS rule_id,
  CASE
    WHEN r.failure_reasons[0] IN ('malformed_json','missing_event_key','orphan_listing_reference',
                                  'orphan_locality_reference','orphan_broker_reference') THEN 'Critical'
    WHEN r.failure_reasons[0] IS NOT NULL THEN 'Major'
  END AS severity,
  COALESCE(NOT r.f_malformed AND (r._rescued_data IS NOT NULL OR COALESCE(r.schema_version <> '1.0', FALSE)), FALSE) AS schema_drift_flag
FROM reasons r
"""
print("Routing rules loaded. Characters:", len(CLASSIFY_SQL))

Routing rules loaded. Characters: 6535


## 12. Step 7 - the batch processor and the runner

For each micro-batch, `process_batch` does four things:

1. Append the raw rows to Bronze with a generated `record_uid` and lineage columns.
2. Read that batch back from Bronze (so every table uses the same `record_uid`).
3. Run the SQL rules.
4. Append the `quarantine` rows to Quarantine **first**, then the `trusted` rows to Silver (the rules read Silver as history, so Silver must not change until the quarantine rows are written).

Every write carries `txnAppId` and `txnVersion = batch_id`, so a retried batch is skipped instead of appended twice.

In [0]:
%python
def _append(df, table, app_id, batch_id):
    (df.write.format("delta")
        .mode("append")
        .option("txnAppId", app_id)
        .option("txnVersion", batch_id)
        .saveAsTable(table))


def process_batch(batch_df, batch_id):
    sp = batch_df.sparkSession

    if batch_df.isEmpty():
        return

    # 1) Bronze: every physical record, nothing filtered
    bronze_df = (
        batch_df
        .withColumn("record_uid", F.expr("concat('STREAM-PHY-', uuid())"))
        .withColumn("batch_id", F.lit(batch_id).cast("bigint"))
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .select(
            "listing_event_id", "listing_id", "event_type", "event_timestamp", "event_sequence",
            "listing_status", "asking_price", "locality_id", "broker_id", "property_type",
            "bedrooms", "area_sqft", "source_system", "schema_version", "ingestion_timestamp",
            "_corrupt_record", "_rescued_data", "source_file_name", "source_file_path",
            "record_uid", "batch_id", "bronze_ingested_at",
        )
    )
    _append(bronze_df, BRONZE, "propiq_stream_bronze", batch_id)

    # 2) Read this batch back from Bronze so all layers share the same record_uid
    sp.sql(f"CREATE OR REPLACE TEMP VIEW stg_batch AS SELECT * FROM {BRONZE} WHERE batch_id = {batch_id}")

    # 3) Apply the SQL routing rules
    sp.sql(CLASSIFY_SQL)

    # 4) Route
    silver_df = sp.sql("""
        SELECT record_uid, listing_event_id, listing_id, event_type, event_timestamp, event_sequence,
               listing_status, asking_price, locality_id, broker_id, property_type, bedrooms, area_sqft,
               source_system, schema_version, ingestion_timestamp,
               schema_drift_flag,
               _rescued_data         AS rescued_data,
               watermark_ts          AS watermark_ts_applied,
               source_file_name, batch_id,
               current_timestamp()   AS processed_at
        FROM stream_batch_classified
        WHERE route = 'trusted'
    """)

    quarantine_df = sp.sql("""
        SELECT record_uid, listing_event_id, listing_id, event_type, event_timestamp, event_sequence,
               listing_status, asking_price, locality_id, broker_id, property_type, bedrooms, area_sqft,
               source_system, schema_version, ingestion_timestamp,
               COALESCE(_corrupt_record,
                        to_json(struct(listing_event_id, listing_id, event_type, event_timestamp, event_sequence,
                                       listing_status, asking_price, locality_id, broker_id, property_type,
                                       bedrooms, area_sqft, source_system, schema_version, ingestion_timestamp))
               )                     AS raw_record,
               _rescued_data         AS rescued_data,
               failure_reasons, primary_reason, rule_id, severity,
               watermark_ts          AS watermark_ts_applied,
               source_file_name, batch_id,
               current_timestamp()   AS quarantined_at
        FROM stream_batch_classified
        WHERE route = 'quarantine'
    """)

    # Quarantine FIRST, then Silver. The rules read Silver as "what is already accepted".
    # If Silver were written first, the quarantine query would see this batch's own rows
    # in Silver and wrongly call every one of them a duplicate. This order also keeps a
    # retried batch correct, because the txnVersion guard skips writes that already landed.
    _append(quarantine_df, QUARANTINE, "propiq_stream_quarantine", batch_id)
    _append(silver_df,     SILVER,     "propiq_stream_silver",     batch_id)


def run_available_now():
    """Process whatever is new in landing/, remember progress in the checkpoint, then stop."""
    query = (
        events_stream.writeStream
            .foreachBatch(process_batch)
            .option("checkpointLocation", CHECKPOINT)
            .queryName("propiq_listing_status_event")
            .trigger(availableNow=True)
            .start()
    )
    query.awaitTermination()
    print("AvailableNow run finished.")

Read `run_available_now()` in plain English:

> Process only the files that are new since the checkpoint, apply the rules, write the three tables, then stop when caught up.

## 13. Drop 01 - normal baseline

### 13.1 Copy Drop 01 into landing

In [0]:
%fs cp /Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_01_normal.json /Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_01_normal.json

True

In [0]:
%sql
LIST '/Volumes/workspace/default/propqi_streaming/landing/';

path,name,size,modification_time
/Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_01_normal.json,listing_status_event_drop_01_normal.json,1725,1791127990000


**Expected result:** only Drop 01 is present. Do not copy Drop 02 yet.

### 13.2 Process the available file

In [0]:
%python
run_available_now()

AvailableNow run finished.


### 13.3 Routing equation for Drop 01

`Bronze physical rows = Trusted rows + Quarantine rows`, so `unaccounted_rows` must be `0`.

In [0]:
%sql
SELECT
  b.source_file_name,
  COUNT(*)                                             AS bronze_physical_rows,
  COUNT(s.record_uid)                                  AS trusted_rows,
  COUNT(q.record_uid)                                  AS quarantine_rows,
  COUNT(*) - COUNT(s.record_uid) - COUNT(q.record_uid) AS unaccounted_rows
FROM workspace.default.bronze_listing_status_stream b
LEFT JOIN workspace.default.silver_listing_status_stream_trusted s ON b.record_uid = s.record_uid
LEFT JOIN workspace.default.quarantine_listing_status_stream     q ON b.record_uid = q.record_uid
GROUP BY b.source_file_name
ORDER BY b.source_file_name;

source_file_name,bronze_physical_rows,trusted_rows,quarantine_rows,unaccounted_rows
listing_status_event_drop_01_normal.json,4,4,0,0


**Expected result:** `bronze_physical_rows` equals the Drop 01 source line count you recorded in step 8.1, and `unaccounted_rows = 0`.

### 13.4 Look at what was accepted

In [0]:
%sql
SELECT listing_event_id, listing_id, event_type, event_timestamp, event_sequence,
       listing_status, asking_price, watermark_ts_applied
FROM workspace.default.silver_listing_status_stream_trusted
ORDER BY event_timestamp, listing_event_id
LIMIT 20;

listing_event_id,listing_id,event_type,event_timestamp,event_sequence,listing_status,asking_price,watermark_ts_applied
EVT-000001,LST-0000001,created,2026-06-25T09:00:00.000Z,1,active,11005988,null
EVT-000002,LST-0000002,price_changed,2026-06-25T09:10:00.000Z,2,active,21357386,null
EVT-000003,LST-0000003,paused,2026-06-25T09:20:00.000Z,2,paused,16873096,null
EVT-000004,LST-0000004,reactivated,2026-06-25T09:30:00.000Z,3,active,21144060,null


`watermark_ts_applied` is empty for Drop 01 because nothing had been accepted before it, so there was no watermark yet.

### 13.5 Prove the checkpoint remembers the file

In [0]:
%sql
SELECT path, create_time, discovery_time, processed_time
FROM cloud_files_state('/Volumes/workspace/default/propqi_streaming/checkpoints/listing_status_event')
ORDER BY path;

path,create_time,discovery_time,processed_time
dbstorage-prod-xekj5baoky/uc/9d3616a7-782e-4597-afd7-856029a35c85/c4529834-236e-463a-b05d-c389a974b8f0/__unitystorage/catalogs/d9c7abde-1595-4371-aae2-ece8219c80f0/volumes/d61884a1-1a1d-42f8-8671-612d8a891d8e/landing/listing_status_event_drop_01_normal.json,2026-10-04T15:33:10.000Z,2026-10-04T15:33:31.673Z,2026-10-04T15:34:03.861Z


**Expected result:** Drop 01 is listed. This is stronger evidence than saying "the checkpoint exists".

## 14. Drop 02 - duplicate event

### 14.1 Copy Drop 02 into landing

In [0]:
%fs cp /Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_02_duplicate.json /Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_02_duplicate.json

True

In [0]:
%sql
LIST '/Volumes/workspace/default/propqi_streaming/landing/';

path,name,size,modification_time
/Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_01_normal.json,listing_status_event_drop_01_normal.json,1725,1791127990000
/Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_02_duplicate.json,listing_status_event_drop_02_duplicate.json,1290,1791128059000


**Expected result:** Drop 01 and Drop 02 are both physically present. The checkpoint, not the folder, is what stops Drop 01 from being processed again.

In [0]:
%python
run_available_now()

AvailableNow run finished.


### 14.2 Routing equation (all files so far)

In [0]:
%sql
SELECT
  b.source_file_name,
  COUNT(*)                                             AS bronze_physical_rows,
  COUNT(s.record_uid)                                  AS trusted_rows,
  COUNT(q.record_uid)                                  AS quarantine_rows,
  COUNT(*) - COUNT(s.record_uid) - COUNT(q.record_uid) AS unaccounted_rows
FROM workspace.default.bronze_listing_status_stream b
LEFT JOIN workspace.default.silver_listing_status_stream_trusted s ON b.record_uid = s.record_uid
LEFT JOIN workspace.default.quarantine_listing_status_stream     q ON b.record_uid = q.record_uid
GROUP BY b.source_file_name
ORDER BY b.source_file_name;

source_file_name,bronze_physical_rows,trusted_rows,quarantine_rows,unaccounted_rows
listing_status_event_drop_01_normal.json,4,4,0,0
listing_status_event_drop_02_duplicate.json,3,2,1,0


**Expected result:** Drop 01 is unchanged (no re-ingestion). Drop 02 has `unaccounted_rows = 0`.

### 14.3 Find repeated event IDs in Bronze

In [0]:
%sql
SELECT listing_event_id, COUNT(*) AS occurrences
FROM workspace.default.bronze_listing_status_stream
GROUP BY listing_event_id
HAVING COUNT(*) > 1
ORDER BY listing_event_id;

listing_event_id,occurrences
EVT-000006,2


Bronze keeps what arrived, including the repeat. This is **business-event duplication**, not file reprocessing.

### 14.4 See how each copy was routed

In [0]:
%sql
SELECT
  b.listing_event_id,
  b.source_file_name,
  b.record_uid,
  CASE WHEN s.record_uid IS NOT NULL THEN 'trusted'
       WHEN q.record_uid IS NOT NULL THEN 'quarantine' END AS route,
  q.failure_reasons
FROM workspace.default.bronze_listing_status_stream b
LEFT JOIN workspace.default.silver_listing_status_stream_trusted s ON b.record_uid = s.record_uid
LEFT JOIN workspace.default.quarantine_listing_status_stream     q ON b.record_uid = q.record_uid
WHERE b.listing_event_id IN (
  SELECT listing_event_id
  FROM workspace.default.bronze_listing_status_stream
  GROUP BY listing_event_id
  HAVING COUNT(*) > 1
)
ORDER BY b.listing_event_id, route;

listing_event_id,source_file_name,record_uid,route,failure_reasons
EVT-000006,listing_status_event_drop_02_duplicate.json,STREAM-PHY-e43b4ba3-4259-4908-9681-f2ce69645909,quarantine,List(duplicate_listing_event_id)
EVT-000006,listing_status_event_drop_02_duplicate.json,STREAM-PHY-1b3ff8b8-eff6-407c-b9ed-b32dd9f8dbb8,trusted,null


**Expected result:** for each repeated ID, exactly one copy is `trusted` and the other is `quarantine` with `duplicate_listing_event_id`.

### 14.5 Trusted Silver must hold each event once

In [0]:
%sql
SELECT listing_event_id, COUNT(*) AS copies_in_trusted
FROM workspace.default.silver_listing_status_stream_trusted
GROUP BY listing_event_id
HAVING COUNT(*) > 1;

listing_event_id,copies_in_trusted


**Expected result:** no rows.

## 15. Drop 03 - late and out-of-order events

This is the playbook's "realistic failure": a late price change arrives after the watermark. The stream must not double-count and must **retain** the late event.

Three words that are easy to confuse:

| Term | Meaning | Control |
|---|---|---|
| **Duplicate** | the same `listing_event_id` arrives again | de-duplication |
| **Late** | the event time is older than the watermark (latest accepted time minus 2 days) | watermark rule |
| **Out-of-order** | the event is inside the watermark but its `event_sequence` does not increase for that listing | sequence rule |

In [0]:
%fs cp /Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_03_late_out_of_order.json /Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_03_late_out_of_order.json

True

In [0]:
%python
run_available_now()

AvailableNow run finished.


### 15.1 What did this drop quarantine, and which watermark was applied?

In [0]:
%sql
SELECT
  listing_event_id, listing_id, event_type, event_timestamp, event_sequence,
  failure_reasons, rule_id, severity, watermark_ts_applied
FROM workspace.default.quarantine_listing_status_stream
WHERE source_file_name LIKE '%drop_03%'
ORDER BY event_timestamp;

listing_event_id,listing_id,event_type,event_timestamp,event_sequence,failure_reasons,rule_id,severity,watermark_ts_applied
EVT-000008,LST-0000008,created,2026-06-21T09:00:00.000Z,1,List(late_event_quarantine),P15-STR-04,Major,2026-06-23T10:05:00.000Z
EVT-000009,LST-0000007,created,2026-06-25T11:05:00.000Z,1,List(sequence_violation),P15-STR-05,Major,2026-06-23T10:05:00.000Z


**Expected behaviour:** an event whose `event_timestamp` is earlier than `watermark_ts_applied` carries `late_event_quarantine`. An event inside the watermark whose sequence went backwards carries `sequence_violation`.

### 15.2 Follow a listing whose sequence was violated

In [0]:
%sql
SELECT 'trusted' AS route, listing_event_id, listing_id, event_sequence, event_timestamp
FROM workspace.default.silver_listing_status_stream_trusted
WHERE listing_id IN (
  SELECT listing_id FROM workspace.default.quarantine_listing_status_stream
  WHERE array_contains(failure_reasons, 'sequence_violation')
)
UNION ALL
SELECT 'quarantine' AS route, listing_event_id, listing_id, event_sequence, event_timestamp
FROM workspace.default.quarantine_listing_status_stream
WHERE listing_id IN (
  SELECT listing_id FROM workspace.default.quarantine_listing_status_stream
  WHERE array_contains(failure_reasons, 'sequence_violation')
)
ORDER BY listing_id, event_timestamp;

route,listing_event_id,listing_id,event_sequence,event_timestamp
trusted,EVT-000007,LST-0000007,3,2026-06-25T11:00:00.000Z
quarantine,EVT-000009,LST-0000007,1,2026-06-25T11:05:00.000Z


Read the timeline top to bottom. The accepted event has the higher sequence; the later-arriving event with a lower sequence is the one that was quarantined.

### 15.3 Routing equation

In [0]:
%sql
SELECT
  b.source_file_name,
  COUNT(*)                                             AS bronze_physical_rows,
  COUNT(s.record_uid)                                  AS trusted_rows,
  COUNT(q.record_uid)                                  AS quarantine_rows,
  COUNT(*) - COUNT(s.record_uid) - COUNT(q.record_uid) AS unaccounted_rows
FROM workspace.default.bronze_listing_status_stream b
LEFT JOIN workspace.default.silver_listing_status_stream_trusted s ON b.record_uid = s.record_uid
LEFT JOIN workspace.default.quarantine_listing_status_stream     q ON b.record_uid = q.record_uid
GROUP BY b.source_file_name
ORDER BY b.source_file_name;

source_file_name,bronze_physical_rows,trusted_rows,quarantine_rows,unaccounted_rows
listing_status_event_drop_01_normal.json,4,4,0,0
listing_status_event_drop_02_duplicate.json,3,2,1,0
listing_status_event_drop_03_late_out_of_order.json,3,1,2,0


## 16. Drop 04 - malformed JSON and an orphan reference

In [0]:
%fs cp /Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_04_malformed_reference.json /Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_04_malformed_reference.json

True

In [0]:
%python
run_available_now()

AvailableNow run finished.


### 16.1 Retained malformed and orphan records

In [0]:
%sql
SELECT
  listing_event_id, listing_id, failure_reasons, rule_id, severity,
  raw_record, source_file_name
FROM workspace.default.quarantine_listing_status_stream
WHERE source_file_name LIKE '%drop_04%'
ORDER BY severity, listing_event_id;

listing_event_id,listing_id,failure_reasons,rule_id,severity,raw_record,source_file_name
null,null,List(malformed_json),P15-STR-01,Critical,"{""listing_event_id"":""EVT-MALFORMED"", bad json",listing_status_event_drop_04_malformed_reference.json
EVT-000011,LST-9999999,List(orphan_listing_reference),P15-DQ-02,Critical,"{""listing_event_id"":""EVT-000011"",""listing_id"":""LST-9999999"",""event_type"":""created"",""event_timestamp"":""2026-06-25T12:05:00.000Z"",""event_sequence"":1,""listing_status"":""active"",""asking_price"":39615810,""locality_id"":""LOC-071"",""broker_id"":""BRK-0201"",""property_type"":""Villa"",""bedrooms"":2,""area_sqft"":3882,""source_system"":""PROPIQ_EVENT_SIM"",""schema_version"":""1.0"",""ingestion_timestamp"":""2026-06-25T12:07:00.000Z""}",listing_status_event_drop_04_malformed_reference.json


**Expected behaviour:** the malformed line is retained in `raw_record` with `malformed_json` (its key fields are null because nothing could be parsed). The orphan event carries `orphan_listing_reference` because its `listing_id` does not exist in `silver_listings_trusted`.

### 16.2 Prove the bad line really exists in the source

The raw text view shows lines whose `listing_event_id` cannot be read as JSON.

In [0]:
%sql
SELECT value AS raw_line
FROM read_files(
  '/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_04_malformed_reference.json',
  format => 'text'
)
WHERE get_json_object(value, '$.listing_event_id') IS NULL;

raw_line
"{""listing_event_id"":""EVT-MALFORMED"", bad json"


### 16.3 Orphan control: left anti-join against the parent master

In [0]:
%sql
SELECT t.listing_event_id, t.listing_id
FROM workspace.default.silver_listing_status_stream_trusted t
LEFT ANTI JOIN workspace.default.silver_listings_trusted p ON t.listing_id = p.listing_id;

listing_event_id,listing_id


**Expected result:** no rows. No orphan may enter Trusted.

## 17. Drop 05 - invalid payload and future event time

In [0]:
%fs cp /Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_05_invalid_payload_future.json /Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_05_invalid_payload_future.json

True

In [0]:
%python
run_available_now()

AvailableNow run finished.


### 17.1 Retained invalid records

In [0]:
%sql
SELECT
  listing_event_id, listing_id, event_timestamp, asking_price, area_sqft,
  failure_reasons, rule_id, severity
FROM workspace.default.quarantine_listing_status_stream
WHERE source_file_name LIKE '%drop_05%'
ORDER BY listing_event_id;

listing_event_id,listing_id,event_timestamp,asking_price,area_sqft,failure_reasons,rule_id,severity
EVT-000013,LST-0000012,2026-06-25T13:05:00.000Z,-5000,1050,List(invalid_price),P15-DQ-03,Major
EVT-000014,LST-0000013,2027-01-01T00:00:00.000Z,2906525,551,List(future_event_time),P15-STR-02,Major


**Expected behaviour:** an out-of-range price or area carries `invalid_price` / `invalid_area`; an event dated in the future carries `future_event_time`.

### 17.2 The future event must not move the watermark

In [0]:
%sql
SELECT
  MAX(event_timestamp)                       AS latest_accepted_event_time,
  MAX(event_timestamp) - INTERVAL 2 DAYS     AS next_watermark
FROM workspace.default.silver_listing_status_stream_trusted;

latest_accepted_event_time,next_watermark
2026-06-25T13:00:00.000Z,2026-06-23T13:00:00.000Z


**Expected result:** the latest accepted time is a real simulation time, not a future date. If the future timestamp had been allowed to set the watermark, almost every later event would wrongly look late.

### 17.3 Routing equation

In [0]:
%sql
SELECT
  b.source_file_name,
  COUNT(*)                                             AS bronze_physical_rows,
  COUNT(s.record_uid)                                  AS trusted_rows,
  COUNT(q.record_uid)                                  AS quarantine_rows,
  COUNT(*) - COUNT(s.record_uid) - COUNT(q.record_uid) AS unaccounted_rows
FROM workspace.default.bronze_listing_status_stream b
LEFT JOIN workspace.default.silver_listing_status_stream_trusted s ON b.record_uid = s.record_uid
LEFT JOIN workspace.default.quarantine_listing_status_stream     q ON b.record_uid = q.record_uid
GROUP BY b.source_file_name
ORDER BY b.source_file_name;

source_file_name,bronze_physical_rows,trusted_rows,quarantine_rows,unaccounted_rows
listing_status_event_drop_01_normal.json,4,4,0,0
listing_status_event_drop_02_duplicate.json,3,2,1,0
listing_status_event_drop_03_late_out_of_order.json,3,1,2,0
listing_status_event_drop_04_malformed_reference.json,3,1,2,0
listing_status_event_drop_05_invalid_payload_future.json,3,1,2,0


## 18. Drop 06 - controlled schema drift

The playbook rule is: **rescue unknown fields; promote only after review.**

In [0]:
%fs cp /Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_06_schema_drift.json /Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_06_schema_drift.json

True

In [0]:
%python
run_available_now()

AvailableNow run finished.


### 18.1 What was rescued?

In [0]:
%sql
SELECT
  listing_event_id,
  schema_version,
  schema_drift_flag,
  rescued_data,
  array_except(map_keys(from_json(rescued_data, 'MAP<STRING,STRING>')), array('_file_path')) AS rescued_field_names
FROM workspace.default.silver_listing_status_stream_trusted
WHERE schema_drift_flag = true
ORDER BY listing_event_id;

listing_event_id,schema_version,schema_drift_flag,rescued_data,rescued_field_names
EVT-000016,2.0,true,"{""marketing_campaign"":""MIDYEAR_2026"",""_file_path"":""/Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_06_schema_drift.json""}",List(marketing_campaign)


**Expected behaviour:** the extra field appears in `rescued_field_names` and its value stays in `rescued_data`. The event is accepted because every contract field is valid. The extra field is **not** a column of Silver.

### 18.2 The drift never changed the contract schema

In [0]:
%sql
DESCRIBE TABLE workspace.default.silver_listing_status_stream_trusted;

col_name,data_type,comment
record_uid,string,null
listing_event_id,string,null
listing_id,string,null
event_type,string,null
event_timestamp,timestamp,null
event_sequence,int,null
listing_status,string,null
asking_price,bigint,null
locality_id,string,null
broker_id,string,null


**Expected result:** the column list is exactly the one created in step 4. No new column appeared.

### 18.3 Promotion is a reviewed decision, not an automatic one

Only after a reviewer approves the new field would you promote it, for example:

```sql
-- DO NOT RUN until the field is reviewed and the contract is updated
-- ALTER TABLE workspace.default.silver_listing_status_stream_trusted ADD COLUMNS (<approved_field> <type>);
```

If you do promote a field, update `streaming/kafka_event_schema.json`, `EVENT_SCHEMA` above and `streaming/structured_streaming_design.md` together, and record the review in the Week Log.

### 18.4 Routing equation

In [0]:
%sql
SELECT
  b.source_file_name,
  COUNT(*)                                             AS bronze_physical_rows,
  COUNT(s.record_uid)                                  AS trusted_rows,
  COUNT(q.record_uid)                                  AS quarantine_rows,
  COUNT(*) - COUNT(s.record_uid) - COUNT(q.record_uid) AS unaccounted_rows
FROM workspace.default.bronze_listing_status_stream b
LEFT JOIN workspace.default.silver_listing_status_stream_trusted s ON b.record_uid = s.record_uid
LEFT JOIN workspace.default.quarantine_listing_status_stream     q ON b.record_uid = q.record_uid
GROUP BY b.source_file_name
ORDER BY b.source_file_name;

source_file_name,bronze_physical_rows,trusted_rows,quarantine_rows,unaccounted_rows
listing_status_event_drop_01_normal.json,4,4,0,0
listing_status_event_drop_02_duplicate.json,3,2,1,0
listing_status_event_drop_03_late_out_of_order.json,3,1,2,0
listing_status_event_drop_04_malformed_reference.json,3,1,2,0
listing_status_event_drop_05_invalid_payload_future.json,3,1,2,0
listing_status_event_drop_06_schema_drift.json,3,3,0,0


## 19. No-new-file rerun - idempotence proof

All six drops are processed. Running the stream again with **no new input** must change nothing.

### 19.1 Capture the state before the rerun

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW rerun_before AS
SELECT
  (SELECT COUNT(*)                         FROM workspace.default.bronze_listing_status_stream)         AS bronze_rows,
  (SELECT COUNT(*)                         FROM workspace.default.silver_listing_status_stream_trusted) AS trusted_rows,
  (SELECT COUNT(*)                         FROM workspace.default.quarantine_listing_status_stream)     AS quarantine_rows,
  (SELECT COUNT(DISTINCT listing_event_id) FROM workspace.default.silver_listing_status_stream_trusted) AS distinct_trusted_event_ids;

SELECT * FROM rerun_before;

bronze_rows,trusted_rows,quarantine_rows,distinct_trusted_event_ids
19,12,7,12


### 19.2 Run again with no new file

In [0]:
%python
run_available_now()

AvailableNow run finished.


### 19.3 Compare before and after

In [0]:
%sql
SELECT
  b.bronze_rows                 AS bronze_before,
  (SELECT COUNT(*) FROM workspace.default.bronze_listing_status_stream)         AS bronze_after,
  b.trusted_rows                AS trusted_before,
  (SELECT COUNT(*) FROM workspace.default.silver_listing_status_stream_trusted) AS trusted_after,
  b.quarantine_rows             AS quarantine_before,
  (SELECT COUNT(*) FROM workspace.default.quarantine_listing_status_stream)     AS quarantine_after,
  b.distinct_trusted_event_ids  AS distinct_ids_before,
  (SELECT COUNT(DISTINCT listing_event_id) FROM workspace.default.silver_listing_status_stream_trusted) AS distinct_ids_after
FROM rerun_before b;

bronze_before,bronze_after,trusted_before,trusted_after,quarantine_before,quarantine_after,distinct_ids_before,distinct_ids_after
19,19,12,12,7,7,12,12


**Expected result:** every `before` equals its `after`. This is the checkpoint lesson in its simplest form.

## 20. Final validation

### 20.1 Source versus Bronze, per file

Every source line must be present in Bronze, including the malformed one.

In [0]:
%sql
WITH src AS (
  SELECT 'listing_status_event_drop_01_normal.json' AS source_file_name, COUNT(*) AS source_lines
  FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_01_normal.json', format => 'text') WHERE trim(value) <> ''
  UNION ALL
  SELECT 'listing_status_event_drop_02_duplicate.json', COUNT(*)
  FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_02_duplicate.json', format => 'text') WHERE trim(value) <> ''
  UNION ALL
  SELECT 'listing_status_event_drop_03_late_out_of_order.json', COUNT(*)
  FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_03_late_out_of_order.json', format => 'text') WHERE trim(value) <> ''
  UNION ALL
  SELECT 'listing_status_event_drop_04_malformed_reference.json', COUNT(*)
  FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_04_malformed_reference.json', format => 'text') WHERE trim(value) <> ''
  UNION ALL
  SELECT 'listing_status_event_drop_05_invalid_payload_future.json', COUNT(*)
  FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_05_invalid_payload_future.json', format => 'text') WHERE trim(value) <> ''
  UNION ALL
  SELECT 'listing_status_event_drop_06_schema_drift.json', COUNT(*)
  FROM read_files('/Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_06_schema_drift.json', format => 'text') WHERE trim(value) <> ''
),
brz AS (
  SELECT source_file_name, COUNT(*) AS bronze_rows
  FROM workspace.default.bronze_listing_status_stream
  GROUP BY source_file_name
)
SELECT
  s.source_file_name,
  s.source_lines,
  b.bronze_rows,
  s.source_lines - COALESCE(b.bronze_rows, 0) AS difference
FROM src s
LEFT JOIN brz b ON s.source_file_name = b.source_file_name
ORDER BY s.source_file_name;

source_file_name,source_lines,bronze_rows,difference
listing_status_event_drop_01_normal.json,4,4,0
listing_status_event_drop_02_duplicate.json,3,3,0
listing_status_event_drop_03_late_out_of_order.json,3,3,0
listing_status_event_drop_04_malformed_reference.json,3,3,0
listing_status_event_drop_05_invalid_payload_future.json,3,3,0
listing_status_event_drop_06_schema_drift.json,3,3,0


**Expected result:** `difference = 0` for all six files.

### 20.2 Reconciliation per file and overall

This is the playbook equation: no failed physical record disappears.

In [0]:
%sql
SELECT
  COALESCE(b.source_file_name, 'ALL FILES')            AS source_file_name,
  COUNT(*)                                             AS bronze_physical_rows,
  COUNT(s.record_uid)                                  AS trusted_rows,
  COUNT(q.record_uid)                                  AS quarantine_rows,
  COUNT(*) - COUNT(s.record_uid) - COUNT(q.record_uid) AS unaccounted_rows
FROM workspace.default.bronze_listing_status_stream b
LEFT JOIN workspace.default.silver_listing_status_stream_trusted s ON b.record_uid = s.record_uid
LEFT JOIN workspace.default.quarantine_listing_status_stream     q ON b.record_uid = q.record_uid
GROUP BY ROLLUP (b.source_file_name)
ORDER BY source_file_name;

source_file_name,bronze_physical_rows,trusted_rows,quarantine_rows,unaccounted_rows
ALL FILES,19,12,7,0
listing_status_event_drop_01_normal.json,4,4,0,0
listing_status_event_drop_02_duplicate.json,3,2,1,0
listing_status_event_drop_03_late_out_of_order.json,3,1,2,0
listing_status_event_drop_04_malformed_reference.json,3,1,2,0
listing_status_event_drop_05_invalid_payload_future.json,3,1,2,0
listing_status_event_drop_06_schema_drift.json,3,3,0,0


**Expected result:** `unaccounted_rows = 0` on every row, including `ALL FILES`.

### 20.3 Control summary - every violation count must be zero

In [0]:
%sql
SELECT 'physical record in BOTH trusted and quarantine' AS control, COUNT(*) AS violations
FROM workspace.default.silver_listing_status_stream_trusted s
JOIN workspace.default.quarantine_listing_status_stream q ON s.record_uid = q.record_uid

UNION ALL
SELECT 'duplicate listing_event_id in trusted', COUNT(*)
FROM (SELECT listing_event_id FROM workspace.default.silver_listing_status_stream_trusted
      GROUP BY listing_event_id HAVING COUNT(*) > 1)

UNION ALL
SELECT 'orphan listing in trusted', COUNT(*)
FROM workspace.default.silver_listing_status_stream_trusted t
LEFT ANTI JOIN workspace.default.silver_listings_trusted p ON t.listing_id = p.listing_id

UNION ALL
SELECT 'late event accepted into trusted', COUNT(*)
FROM workspace.default.silver_listing_status_stream_trusted
WHERE watermark_ts_applied IS NOT NULL AND event_timestamp < watermark_ts_applied

UNION ALL
SELECT 'event_sequence not increasing within listing_id in trusted', COUNT(*)
FROM (
  SELECT event_sequence,
         LAG(event_sequence) OVER (PARTITION BY listing_id ORDER BY event_timestamp, listing_event_id) AS prev_sequence
  FROM workspace.default.silver_listing_status_stream_trusted
)
WHERE prev_sequence IS NOT NULL AND event_sequence <= prev_sequence

UNION ALL
SELECT 'price or area outside contract in trusted', COUNT(*)
FROM workspace.default.silver_listing_status_stream_trusted
WHERE asking_price NOT BETWEEN 2000000 AND 150000000 OR area_sqft NOT BETWEEN 300 AND 8000

UNION ALL
SELECT 'future-dated event in trusted', COUNT(*)
FROM workspace.default.silver_listing_status_stream_trusted
WHERE event_timestamp > current_timestamp() OR event_timestamp > TIMESTAMP '2026-12-31 23:59:59';

control,violations
physical record in BOTH trusted and quarantine,0
duplicate listing_event_id in trusted,0
orphan listing in trusted,0
late event accepted into trusted,0
event_sequence not increasing within listing_id in trusted,0
price or area outside contract in trusted,0
future-dated event in trusted,0


**Expected result:** `violations = 0` on every line. Any non-zero line is a blocker: inspect it, fix the rule or the data, and re-run from a clean reset.

### 20.4 Every quarantine reason, per file

A record can carry several reasons, so each reason is counted separately here.

In [0]:
%sql
SELECT source_file_name, reason, COUNT(*) AS records
FROM (
  SELECT source_file_name, explode(failure_reasons) AS reason
  FROM workspace.default.quarantine_listing_status_stream
)
GROUP BY source_file_name, reason
ORDER BY source_file_name, reason;

source_file_name,reason,records
listing_status_event_drop_02_duplicate.json,duplicate_listing_event_id,1
listing_status_event_drop_03_late_out_of_order.json,late_event_quarantine,1
listing_status_event_drop_03_late_out_of_order.json,sequence_violation,1
listing_status_event_drop_04_malformed_reference.json,malformed_json,1
listing_status_event_drop_04_malformed_reference.json,orphan_listing_reference,1
listing_status_event_drop_05_invalid_payload_future.json,future_event_time,1
listing_status_event_drop_05_invalid_payload_future.json,invalid_price,1


This is your scenario-coverage matrix: duplicate, late, out-of-order, malformed, orphan, invalid and drift should each be traceable to the drop that was designed to test it.

### 20.5 Quarantine by primary reason, rule and severity

In [0]:
%sql
SELECT primary_reason, rule_id, severity, COUNT(*) AS records
FROM workspace.default.quarantine_listing_status_stream
GROUP BY primary_reason, rule_id, severity
ORDER BY severity, primary_reason;

primary_reason,rule_id,severity,records
malformed_json,P15-STR-01,Critical,1
orphan_listing_reference,P15-DQ-02,Critical,1
duplicate_listing_event_id,P15-STR-03,Major,1
future_event_time,P15-STR-02,Major,1
invalid_price,P15-DQ-03,Major,1
late_event_quarantine,P15-STR-04,Major,1
sequence_violation,P15-STR-05,Major,1


### 20.6 Event time versus ingestion time

In [0]:
%sql
SELECT listing_event_id, event_type, event_timestamp, ingestion_timestamp, bronze_ingested_at, source_file_name
FROM workspace.default.bronze_listing_status_stream
ORDER BY bronze_ingested_at, event_timestamp
LIMIT 20;

listing_event_id,event_type,event_timestamp,ingestion_timestamp,bronze_ingested_at,source_file_name
EVT-000001,created,2026-06-25T09:00:00.000Z,2026-06-25T09:02:00.000Z,2026-10-04T15:33:36.839Z,listing_status_event_drop_01_normal.json
EVT-000002,price_changed,2026-06-25T09:10:00.000Z,2026-06-25T09:12:00.000Z,2026-10-04T15:33:36.839Z,listing_status_event_drop_01_normal.json
EVT-000003,paused,2026-06-25T09:20:00.000Z,2026-06-25T09:22:00.000Z,2026-10-04T15:33:36.839Z,listing_status_event_drop_01_normal.json
EVT-000004,reactivated,2026-06-25T09:30:00.000Z,2026-06-25T09:32:00.000Z,2026-10-04T15:33:36.839Z,listing_status_event_drop_01_normal.json
EVT-000005,price_changed,2026-06-25T10:00:00.000Z,2026-06-25T10:02:00.000Z,2026-10-04T15:34:33.788Z,listing_status_event_drop_02_duplicate.json
EVT-000006,paused,2026-06-25T10:05:00.000Z,2026-06-25T10:07:00.000Z,2026-10-04T15:34:33.788Z,listing_status_event_drop_02_duplicate.json
EVT-000006,paused,2026-06-25T10:05:00.000Z,2026-06-25T10:07:00.000Z,2026-10-04T15:34:33.788Z,listing_status_event_drop_02_duplicate.json
EVT-000008,created,2026-06-21T09:00:00.000Z,2026-06-21T09:02:00.000Z,2026-10-04T15:35:22.955Z,listing_status_event_drop_03_late_out_of_order.json
EVT-000007,price_changed,2026-06-25T11:00:00.000Z,2026-06-25T11:02:00.000Z,2026-10-04T15:35:22.955Z,listing_status_event_drop_03_late_out_of_order.json
EVT-000009,created,2026-06-25T11:05:00.000Z,2026-06-25T11:07:00.000Z,2026-10-04T15:35:22.955Z,listing_status_event_drop_03_late_out_of_order.json


Explain the three clocks:

- `event_timestamp` is when the listing event happened (the watermark uses this);
- `ingestion_timestamp` is when the source system recorded it;
- `bronze_ingested_at` is when Databricks wrote it.

## 21. Delta history and checkpoint state as evidence

Use your own Databricks output. Do not fabricate version numbers or timestamps.

In [0]:
%sql
DESCRIBE HISTORY workspace.default.bronze_listing_status_stream;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-10-04T15:37:32.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,5,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 7834)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
5,2026-10-04T15:36:51.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 7250)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
4,2026-10-04T15:36:10.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 7234)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
3,2026-10-04T15:35:24.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 7201)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
2,2026-10-04T15:34:35.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 7104)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
1,2026-10-04T15:33:40.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,0,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 7113)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
0,2026-10-04T15:33:00.000Z,71736716437240,thota.madhulika05@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> PropIQ Week 10 streaming Bronze: every physical listing-status record, raw-preserving, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-4d8de2f6-0e09-4f13-8fd5-fa17e7b0e065"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-5ebf576f-b23c-4605-92fd-dc15c9982464""}, statsOnLoad -> false)",null,List(2332163692648232),6ed1be30-aa30-437f-8641-2a62a2437096,1004-153136-yfbup2bg-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13


In [0]:
%sql
DESCRIBE HISTORY workspace.default.silver_listing_status_stream_trusted;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-10-04T15:37:47.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,5,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 3, numOutputBytes -> 7419)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
5,2026-10-04T15:37:07.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,4,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 6235)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
4,2026-10-04T15:36:26.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,3,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 6185)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
3,2026-10-04T15:35:44.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 6188)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
2,2026-10-04T15:34:55.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 6433)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
1,2026-10-04T15:34:03.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numOutputRows -> 4, numOutputBytes -> 6647)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
0,2026-10-04T15:33:02.000Z,71736716437240,thota.madhulika05@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> PropIQ Week 10 streaming Trusted Silver: accepted listing-status events only, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e1c837f2-2644-4309-91a5-c3d03892369d"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-8ea5b589-c72c-4c84-acb4-584cc600448a""}, statsOnLoad -> false)",null,List(2332163692648232),05772226-4173-4a37-b526-abeb21cbe048,1004-153136-yfbup2bg-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13


In [0]:
%sql
DESCRIBE HISTORY workspace.default.quarantine_listing_status_stream;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2026-10-04T15:37:41.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,5,WriteSerializable,true,"Map(numFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
5,2026-10-04T15:37:00.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,4,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 9653)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
4,2026-10-04T15:36:20.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,3,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 8709)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
3,2026-10-04T15:35:34.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,2,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 2, numOutputBytes -> 9635)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
2,2026-10-04T15:34:46.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> false, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,1,WriteSerializable,true,"Map(numFiles -> 1, numOutputRows -> 1, numOutputBytes -> 9268)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
1,2026-10-04T15:33:53.000Z,71736716437240,thota.madhulika05@gmail.com,WRITE,"Map(mode -> Append, statsOnLoad -> true, partitionBy -> [])",null,List(2332163692648232),null,1004-153136-yfbup2bg-v2n,0,WriteSerializable,true,"Map(numFiles -> 0, numOutputRows -> 0, numOutputBytes -> 0)",null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13
0,2026-10-04T15:33:04.000Z,71736716437240,thota.madhulika05@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> PropIQ Week 10 streaming quarantine: every failed physical record with rule evidence, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-f9606e46-b4af-4fe4-b697-ca94a651fad1"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-6d4c498c-b17b-44c6-ab2b-f4d660e06170""}, statsOnLoad -> false)",null,List(2332163692648232),b9ddab04-fa4f-4ab7-a8c6-2b5424c1ae36,1004-153136-yfbup2bg-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.8.x-aarch64-photon-scala2.13


In [0]:
%sql
SELECT path, processed_time
FROM cloud_files_state('/Volumes/workspace/default/propqi_streaming/checkpoints/listing_status_event')
ORDER BY path;

path,processed_time
dbstorage-prod-xekj5baoky/uc/9d3616a7-782e-4597-afd7-856029a35c85/c4529834-236e-463a-b05d-c389a974b8f0/__unitystorage/catalogs/d9c7abde-1595-4371-aae2-ece8219c80f0/volumes/d61884a1-1a1d-42f8-8671-612d8a891d8e/landing/listing_status_event_drop_01_normal.json,2026-10-04T15:34:03.861Z
dbstorage-prod-xekj5baoky/uc/9d3616a7-782e-4597-afd7-856029a35c85/c4529834-236e-463a-b05d-c389a974b8f0/__unitystorage/catalogs/d9c7abde-1595-4371-aae2-ece8219c80f0/volumes/d61884a1-1a1d-42f8-8671-612d8a891d8e/landing/listing_status_event_drop_02_duplicate.json,2026-10-04T15:34:55.324Z
dbstorage-prod-xekj5baoky/uc/9d3616a7-782e-4597-afd7-856029a35c85/c4529834-236e-463a-b05d-c389a974b8f0/__unitystorage/catalogs/d9c7abde-1595-4371-aae2-ece8219c80f0/volumes/d61884a1-1a1d-42f8-8671-612d8a891d8e/landing/listing_status_event_drop_03_late_out_of_order.json,2026-10-04T15:35:44.081Z
dbstorage-prod-xekj5baoky/uc/9d3616a7-782e-4597-afd7-856029a35c85/c4529834-236e-463a-b05d-c389a974b8f0/__unitystorage/catalogs/d9c7abde-1595-4371-aae2-ece8219c80f0/volumes/d61884a1-1a1d-42f8-8671-612d8a891d8e/landing/listing_status_event_drop_04_malformed_reference.json,2026-10-04T15:36:26.831Z
dbstorage-prod-xekj5baoky/uc/9d3616a7-782e-4597-afd7-856029a35c85/c4529834-236e-463a-b05d-c389a974b8f0/__unitystorage/catalogs/d9c7abde-1595-4371-aae2-ece8219c80f0/volumes/d61884a1-1a1d-42f8-8671-612d8a891d8e/landing/listing_status_event_drop_05_invalid_payload_future.json,2026-10-04T15:37:06.998Z
dbstorage-prod-xekj5baoky/uc/9d3616a7-782e-4597-afd7-856029a35c85/c4529834-236e-463a-b05d-c389a974b8f0/__unitystorage/catalogs/d9c7abde-1595-4371-aae2-ece8219c80f0/volumes/d61884a1-1a1d-42f8-8671-612d8a891d8e/landing/listing_status_event_drop_06_schema_drift.json,2026-10-04T15:37:47.930Z


**Expected result:** all six drop files appear in the checkpoint state. This answers: *"How do we know the stream remembers which files it has processed?"*

## 22. Stretch: drop the same file again (optional)

The playbook failure scenario says the same event file is dropped again. A re-dropped file has a **new file name**, so Auto Loader treats it as new input. The event IDs, however, were already accepted, so the rule must quarantine them instead of double-counting.

> Run this **after** you have captured the main evidence. It adds a seventh file to Bronze, so the totals above will change. Reset afterwards if you want a clean final demonstration.

In [0]:
%fs cp /Volumes/workspace/default/propqi_streaming/source/listing_status_event_drop_02_duplicate.json /Volumes/workspace/default/propqi_streaming/landing/listing_status_event_drop_02_duplicate_REPLAY.json

True

In [0]:
%python
run_available_now()

AvailableNow run finished.


In [0]:
%sql
SELECT
  b.source_file_name,
  COUNT(*)                                             AS bronze_physical_rows,
  COUNT(s.record_uid)                                  AS trusted_rows,
  COUNT(q.record_uid)                                  AS quarantine_rows
FROM workspace.default.bronze_listing_status_stream b
LEFT JOIN workspace.default.silver_listing_status_stream_trusted s ON b.record_uid = s.record_uid
LEFT JOIN workspace.default.quarantine_listing_status_stream     q ON b.record_uid = q.record_uid
WHERE b.source_file_name LIKE '%REPLAY%'
GROUP BY b.source_file_name;

source_file_name,bronze_physical_rows,trusted_rows,quarantine_rows
listing_status_event_drop_02_duplicate_REPLAY.json,3,0,3


**Expected result:** the replayed file adds Bronze rows but `trusted_rows = 0`. Trusted Silver is unchanged, so nothing was double-counted.

## 23. Common failures and recovery

| Symptom | Likely cause | Inspect | Recovery |
|---|---|---|---|
| Counts unexpectedly increase after a rerun | table appended without the checkpoint, or checkpoint changed | `DESCRIBE HISTORY`, `cloud_files_state` | restore the stable checkpoint, or do a documented clean reset |
| Stream reprocesses old files | checkpoint deleted or path changed | the `CHECKPOINT` constant | use the locked path; reset only in a documented clean test |
| Malformed JSON is lost | rescue/corrupt column not retained | Bronze `_corrupt_record`, raw text query in 16.2 | keep the rescued payload plus source filename and quarantine reason |
| Everything is quarantined as `orphan_*` | parent masters do not match the events | dry-run check in 8.3 | correct the source or masters; record a blocker. Never loosen the rule to hide it |
| Good events look late | a future or invalid event moved the watermark | `latest_accepted_event_time` query in 17.2 | confirm only accepted events feed the watermark |
| `unaccounted_rows` is not 0, or every row is quarantined as duplicate | a record was written to Bronze but not routed, or Silver was written before the rules finished reading it | routing equation, batch ids, write order in `process_batch` | keep Quarantine-before-Silver; re-run from a clean reset |
| Retried batch looks skipped | `txnVersion` already recorded for that batch | `DESCRIBE HISTORY` | expected after a failure; do not delete the checkpoint without resetting all three tables |
| Foreign key lookups fail with "table not found" | masters are in another schema | step 6.1 | update the three parent table names in the rules |
| Evidence is only a screenshot | working artifact not committed | repo files and history | commit the notebook and validation; keep screenshots as support only |

## 24. Controlled reset

Use this only before a completely fresh demonstration. **Reset all three tables, the landing folder and the checkpoint together.** Resetting only the checkpoint reprocesses files into non-empty tables; resetting only the tables leaves a checkpoint that thinks every file is done.

The six files in `source/` stay untouched. The parent masters (`silver_listings_trusted` and the others) are never touched.

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.bronze_listing_status_stream;
DROP TABLE IF EXISTS workspace.default.silver_listing_status_stream_trusted;
DROP TABLE IF EXISTS workspace.default.quarantine_listing_status_stream;

In [0]:
%fs rm -r /Volumes/workspace/default/propqi_streaming/landing/

True

In [0]:
%fs rm -r /Volumes/workspace/default/propqi_streaming/checkpoints/

True

In [0]:
%fs mkdirs /Volumes/workspace/default/propqi_streaming/landing/

True

In [0]:
%fs mkdirs /Volumes/workspace/default/propqi_streaming/checkpoints/

True

After a reset, re-run step 4 (create the tables) and restart from Drop 01.

## 25. Evidence, ownership and GitHub

### Required evidence (your own output)

- `LIST` of `source/` showing six files, and of `landing/` empty before Drop 01;
- source control totals from 8.1 and the Drop 01 dry-run reference check;
- routing equation after each drop;
- Drop 02 duplicate routing (14.4) and the empty duplicate check (14.5);
- Drop 03 late and sequence evidence with `watermark_ts_applied`;
- Drop 04 retained malformed line and orphan, plus the empty anti-join;
- Drop 05 invalid and future events, plus the watermark check;
- Drop 06 rescued field names and the unchanged Silver schema;
- no-new-file before/after comparison;
- final reconciliation, control summary and reason matrix;
- `DESCRIBE HISTORY` and `cloud_files_state` output;
- `weekly_logs/week10_log.md` and the AI Transparency Note.

### Repository files to update this week

| Path | What to record |
|---|---|
| `notebooks/07_streaming_simulation.ipynb` | this notebook with your executed outputs |
| `data_sample/streaming/` | the six drops used |
| `streaming/kafka_event_schema.json` | the event contract mirrored from `EVENT_SCHEMA` |
| `streaming/structured_streaming_design.md` | controls, rule table (including the notebook-local `P15-STR-*` IDs) and design decisions |
| `weekly_logs/week10_log.md` | outcome, files/objects, validation, blockers/rework, individual contribution, AI Transparency Note |

Commit examples:

- `week10: implement controlled streaming simulation`
- `week10: add PropIQ validation and reconciliation evidence`
- `week10: close mentor rework and document recovery`

### Team ownership (Student A / B / C)

| Student | Deep ownership this week | Must explain |
|---|---|---|
| A | steps 1 to 5: source, landing, tables, Auto Loader; backup B | input grain, source contract, first validation |
| B | steps 6 to 7: routing rules and batch processor; backup C | rule logic, failure behaviour, reconciliation |
| C | validation, evidence and demo; backup A | consumer impact, GitHub proof, limitation |

All three students should be able to explain the complete flow.

## 26. Exit checklist

- [ ] I can explain source, landing and checkpoint.
- [ ] Six drops were processed one at a time, in order, from an empty landing folder.
- [ ] Trusted Silver contains each `listing_event_id` at most once.
- [ ] Late, out-of-order, malformed, orphan, invalid, future and drifted records are each retained and explained.
- [ ] `Bronze = Trusted + Quarantine` for every file, with `unaccounted_rows = 0`.
- [ ] The no-new-file rerun changed nothing.
- [ ] The schema drift was rescued and **not** promoted.
- [ ] I captured real Databricks evidence and committed the notebook.
- [ ] I can explain one failure and its recovery.
- [ ] Week Log and AI Transparency Note are complete; mentor rework is closed or recorded as a blocker.

## 27. Viva

**Official question:** what is the difference between a late event, a duplicate event and an out-of-order event?

Also be ready to answer:

1. Why does Bronze keep malformed and duplicate rows?
2. Why is the late event retained instead of dropped by Spark's watermark?
3. Why must only accepted events move the watermark?
4. What does the checkpoint remember, and what happens if it is deleted?
5. Why is a re-dropped file with a new name not a checkpoint problem?
6. What does `_rescued_data` hold, and why is schema drift not promoted automatically?
7. Why do orphan checks read Trusted Silver and not Bronze?
8. What do `txnAppId` and `txnVersion` protect against?
9. How does `event_timestamp` differ from `ingestion_timestamp` and `bronze_ingested_at`?
10. What must change if the event contract gains a new approved field?

### Self-check hints

- **Duplicate:** same `listing_event_id`, seen again. **Late:** event time older than the watermark. **Out-of-order:** inside the watermark, but the sequence for that listing goes backwards.
- The checkpoint remembers which input files were already processed; delete it and old files are read again.

### Boundary

**Week 10:** controlled streaming simulation and proof.  
**Week 11:** integrate the whole platform, close cross-layer issues and rehearse.  
**Week 12:** submit, present and defend.

---
# PART 2: Live Events - Continuous Streaming Demonstration

**Goal:** Understand the difference between file arrival (Part 1 - Auto Loader) and live business-event arrival.

## What Part 2 demonstrates

| Component | Purpose |
|---|---|
| **Live source table** | `pageloop_live_events` - a Delta table that receives new listing status events continuously |
| **Python producer** | Small simulator that generates CHECKOUT / RENEWED / RETURNED events every few seconds |
| **FROM STREAM consumer** | SQL streaming query that reads new events incrementally from the source |
| **Observation** | Run a count query repeatedly: count grows while producer runs, stops when producer stops |

### Key difference from Part 1

Part 1 (Auto Loader) watched a **folder** for new **files**. Part 2 watches a **Delta table** for new **rows**. The business events themselves are still listing status changes, but they arrive row-by-row rather than in batch file drops.

---